In [ ]:
import geopandas as gpd
import zipfile
from pathlib import Path
from shapely.ops import substring, linemerge
from src.load_data import *
import tempfile
import shutil
import numpy as np
import geopandas as gpd

In [ ]:
data = load_var(subject_id=1, variable="CO2")
data

In [ ]:
df = load_subject(1)
df.dropna(inplace=True)

import pandas as pd

# Load and clean
df = load_subject(1)
df.dropna(inplace=True)

# Compare all columns pairwise
duplicates_by_value = []

cols = df.columns
for i in range(len(cols)):
    for j in range(i + 1, len(cols)):
        if df[cols[i]].equals(df[cols[j]]):
            duplicates_by_value.append((cols[i], cols[j]))

# Show result
if duplicates_by_value:
    print("Columns with identical values:")
    for pair in duplicates_by_value:
        print(f"  • {pair[0]} == {pair[1]}")
else:
    print("No columns with identical values.")


In [ ]:
df

In [ ]:
kmz_path = 'data/raw_data/route.kmz'

# Create a temporary directory
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir_path = Path(tmpdir)

    # Extract the KMZ to the temp folder
    with zipfile.ZipFile(kmz_path, 'r') as z:
        z.extractall(tmpdir_path)

    # Read the KML file (usually named 'doc.kml')
    kml_path = tmpdir_path / 'doc.kml'
    gdf = gpd.read_file(kml_path, driver='KML')

In [ ]:
gdf

In [ ]:
route = gdf['geometry'][0]
points = gdf['geometry'][1:]

In [ ]:
# 1) grab your closed LineString
line = gdf.geometry.iloc[0]

# 2) extract & dedupe your 8 “cut” points, then project & sort them
pts = gdf.geometry.iloc[1:].drop_duplicates().to_list()
dists = sorted(line.project(pt) for pt in pts)

# 3) build the wrap‑around piece (last → first)
wrap = linemerge([
    substring(line, dists[-1], line.length),
    substring(line, 0.0,      dists[0])
])

# 4) build the in‑between pieces (1→2, 2→3, …)
segs = [wrap] + [
    substring(line, a, b)
    for a, b in zip(dists[:-1], dists[1:])
]

# 5) turn into a GeoSeries
segments_series = gpd.GeoSeries(segs, crs=gdf.crs)


In [ ]:
# create another column in a geodataframe segs
segments_df = gpd.GeoDataFrame({'geometry': segments_series})
segments_df['location'] = ["FG", "GH", "AB", "BC", "CD", "DE", "EF"]
segments_df

In [ ]:
S_dynamic = data[data["regime"] == "dynamic"]
S_dynamic

In [ ]:
# 1) Merge S_dynamic with your segments GeoDataFrame
#    (assumes segments_df has columns ['location','geometry'])
sd = S_dynamic.merge(
    segments_df[['location','geometry']],
    on='location',
    how='left'
)
sd.index = S_dynamic.index

# 2) Prepare an empty column for the interpolated Points
sd['sample_pt'] = None
# 3) For each segment, evenly space the points
for loc, group in sd.groupby('location'):
    seg = group.geometry.iloc[0]         # the LineString for this location
    L   = seg.length                     # its total length
    N   = len(group)                     # how many samples to place

    # choose your scheme:
    #  - including endpoints: np.linspace(0   , L, N)
    #  - *excluding* endpoints:     np.linspace(0.5, L-0.5, N) /or/ (np.arange(N)+0.5)/N * L
    dists = np.linspace(0, L, N)         # here we *include* start & end

    # interpolate each distance into a Point
    pts = [seg.interpolate(d) for d in dists]

    # write them back into sd, preserving original row‐order
    sd.loc[group.index, 'sample_pt'] = pts


# 4) Extract X/Y (and Z, if present)
sd['x'] = sd['sample_pt'].apply(lambda p: p.x)
sd['y'] = sd['sample_pt'].apply(lambda p: p.y)
# if 3D:
# sd['z'] = sd['sample_pt'].apply(lambda p: p.z)

# Inspect
print(sd[['location','sample_pt','x','y']].head())


In [ ]:
sd

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
for loc, group in sd.groupby('location'):
    ax.scatter(group['x'], group['y'], label=loc)
ax.set_xlabel('X coordinate')
ax.set_ylabel('Y coordinate')
ax.set_title('Interpolated Points by Segment Location')
ax.legend(title='Location', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
points_df = gdf.drop(0)
points_df["location"] = points_df["Name"].str[0]
points_df["x"] = points_df.geometry.x
points_df["y"] = points_df.geometry.y
points_df.drop(columns=["Name", "Description", "geometry"], inplace=True)
points_df

In [ ]:
# 1) First, drop any duplicate locations in your points_df so join doesn’t blow up:
points_unique = points_df.drop_duplicates(subset="location").set_index("location")

# 2) Select your static regime rows
S_static = data.loc[data["regime"] == "static"]

# 3) Join on the ‘location’ index of points_unique
S_static = S_static.join(points_unique, on="location")
S_static

In [ ]:
S_dynamic = sd.drop(columns=["sample_pt", "geometry"])
S_dynamic

In [ ]:
S = pd.concat([S_static, S_dynamic])
S.sort_index(inplace=True)
S.to_parquet("data/processed_data/S1CO2-approx-coordinates.parquet", index=False)
S